In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ethereum-frauddetection-dataset/transaction_dataset.csv


In [2]:
import pandas as pd

# Load dataset (replace with your actual file path)
df = pd.read_csv("/kaggle/input/ethereum-frauddetection-dataset/transaction_dataset.csv")

# Quick look at the data
print(df.shape)
print(df.head())

# Check for missing values
print(df.isnull().sum())

# Fraud vs non-fraud distribution
print(df["FLAG"].value_counts(normalize=True))


(9841, 51)
   Unnamed: 0  Index                                     Address  FLAG  \
0           0      1  0x00009277775ac7d0d59eaad8fee3d10ac6c805e8     0   
1           1      2  0x0002b44ddb1476db43c868bd494422ee4c136fed     0   
2           2      3  0x0002bda54cb772d040f779e88eb453cac0daa244     0   
3           3      4  0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e     0   
4           4      5  0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89     0   

   Avg min between sent tnx  Avg min between received tnx  \
0                    844.26                       1093.71   
1                  12709.07                       2958.44   
2                 246194.54                       2434.02   
3                  10219.60                      15785.09   
4                     36.61                      10707.77   

   Time Diff between first and last (Mins)  Sent tnx  Received Tnx  \
0                                704785.63       721            89   
1                               12182

In [3]:
# Basic libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


In [4]:
# Load dataset
df = pd.read_csv("/kaggle/input/ethereum-frauddetection-dataset/transaction_dataset.csv")

# Show basic info
print("Shape of dataset:", df.shape)
print("\nColumns:\n", df.columns)
print("\nFirst 5 rows:\n")
print(df.head())


Shape of dataset: (9841, 51)

Columns:
 Index(['Unnamed: 0', 'Index', 'Address', 'FLAG', 'Avg min between sent tnx',
       'Avg min between received tnx',
       'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx',
       'Number of Created Contracts', 'Unique Received From Addresses',
       'Unique Sent To Addresses', 'min value received', 'max value received ',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'min value sent to contract', 'max val sent to contract',
       'avg value sent to contract',
       'total transactions (including tnx to create contract',
       'total Ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance',
       ' Total ERC20 tnxs', ' ERC20 total Ether received',
       ' ERC20 total ether sent', ' ERC20 total Ether sent contract',
       ' ERC20 uniq sent addr', ' ERC20 uniq rec addr',
       ' ERC20 uniq sent addr.1', ' ERC20 uniq rec contract addr',
       ' ER

In [5]:
# Clean column names (strip spaces and replace with underscores)
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Check the cleaned column names
print(df.columns.tolist()[:15])  # print first 15 to verify


['Unnamed:_0', 'Index', 'Address', 'FLAG', 'Avg_min_between_sent_tnx', 'Avg_min_between_received_tnx', 'Time_Diff_between_first_and_last_(Mins)', 'Sent_tnx', 'Received_Tnx', 'Number_of_Created_Contracts', 'Unique_Received_From_Addresses', 'Unique_Sent_To_Addresses', 'min_value_received', 'max_value_received', 'avg_val_received']


In [6]:
# Fill NaN with 'Unknown'
df['ERC20_most_sent_token_type'] = df['ERC20_most_sent_token_type'].fillna('Unknown')
df['ERC20_most_rec_token_type'] = df['ERC20_most_rec_token_type'].fillna('Unknown')

# Encode categorical values
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['ERC20_most_sent_token_type'] = le.fit_transform(df['ERC20_most_sent_token_type'])
df['ERC20_most_rec_token_type'] = le.fit_transform(df['ERC20_most_rec_token_type'])

# Separate features and target
X = df.drop(columns=['FLAG'])
y = df['FLAG']

print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts())


Features shape: (9841, 50)
Target distribution:
 FLAG
0    7662
1    2179
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split

# Split the dataset (stratify ensures class ratio stays same in train/test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Fraud proportion in training:", y_train.mean())
print("Fraud proportion in test:", y_test.mean())


Training set shape: (7872, 50)
Test set shape: (1969, 50)
Fraud proportion in training: 0.22141768292682926
Fraud proportion in test: 0.22143219908583037


In [8]:
from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Fit on training data, transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training set shape:", X_train_scaled.shape)
print("Scaled test set shape:", X_test_scaled.shape)


ValueError: could not convert string to float: '0x7ff05442651fc3daa6d6a06f64cba30cad7b68b9'

In [ ]:
# Separate numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Fill missing values
X[num_cols] = X[num_cols].fillna(0)  # or X[num_cols].fillna(X[num_cols].median())
X[cat_cols] = X[cat_cols].fillna('Unknown')

# Check again
print("Missing values after imputation:\n", X.isna().sum().sum())


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1️⃣ Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2️⃣ Create the Random Forest model
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)

# 3️⃣ Train the model
rf_model.fit(X_train, y_train)

# 4️⃣ Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:,1]  # probability for ROC-AUC

# 5️⃣ Evaluate
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))


In [ ]:
import joblib

# Save the model
joblib.dump(rf_model, 'fraud_detection_rf_model.pkl')

print("Random Forest fraud detection model saved successfully!")


In [ ]:
# Load the model
rf_model_loaded = joblib.load('fraud_detection_rf_model.pkl')
